In [19]:
import os                    # File path operations (cross-platform: Windows/macOS/Linux)
import re                    # Regular expressions for text cleaning
import time                  # Timing training epochs
import math                  # math.exp() for perplexity calculation
import random                # Python's built-in RNG — we'll seed this
import urllib.request        # Download dataset without extra dependencies
import json                  # Potentially save/load results
from collections import Counter  # Count word frequencies for vocabulary

# ── Numerical & Deep Learning ──────────────────────────────────────────────
import numpy as np           # Array operations; we also seed this
import torch                 # PyTorch deep learning framework
import torch.nn as nn        # Neural network building blocks (layers, losses)
import torch.optim as optim  # Optimization algorithms (Adam, SGD, etc.)
from torch.utils.data import Dataset, DataLoader  # Data pipeline abstraction

# ── Visualization ──────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt   # Plotting library for all charts
import matplotlib.gridspec as gridspec  # Advanced subplot layouts
matplotlib.rcParams['figure.dpi'] = 120   # Sharper figures in notebooks

print(" All libraries imported successfully")
print(f"   PyTorch version : {torch.__version__}")
print(f"   NumPy version   : {np.__version__}")

 All libraries imported successfully
   PyTorch version : 2.10.0+cu126
   NumPy version   : 2.3.5


In [20]:
# ── Fix ALL Random Seeds for Full Reproducibility ─────────────────────────
SEED = 42

random.seed(SEED)           # Python built-in RNG (used by some data ops)
np.random.seed(SEED)        # NumPy RNG (used for train/val splits)
torch.manual_seed(SEED)     # PyTorch CPU RNG
torch.cuda.manual_seed_all(SEED)  # PyTorch GPU RNG (all GPUs, harmless if no GPU)

# Make convolution algorithms deterministic (minor speed trade-off)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False  # Disable auto-tuner for reproducibility

print(f"All random seeds fixed to: {SEED}")

All random seeds fixed to: 42


In [21]:
# ── Device Detection: M4 Mac / NVIDIA GPU / CPU ───────────────────────────
def get_device():
    """
    Detect the best available hardware accelerator.
    Priority: CUDA (NVIDIA) > MPS (Apple Silicon) > CPU
    """
    if torch.cuda.is_available():
        device = torch.device("cuda")
        name = torch.cuda.get_device_name(0)
        print(f" Using CUDA GPU: {name}")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
        print(" Using Apple MPS (Metal Performance Shaders) — M-series chip detected")
    else:
        device = torch.device("cpu")
        print(" Using CPU (no GPU acceleration available)")
    return device

DEVICE = get_device()
print(f"   Device object: {DEVICE}")

 Using CUDA GPU: NVIDIA GeForce RTX 4060
   Device object: cuda


In [22]:
# ── Hyperparameters: Shared Across RNN / LSTM / GRU ──────────────────────

# --- Data ---
VOCAB_SIZE_LIMIT = 5000       # Keep only the N most frequent words
SEQ_LEN         = 30          # Input sequence length (sliding window size)
BATCH_SIZE      = 64          # Number of sequences per training step
VAL_SPLIT       = 0.1         # 10% of data held out for validation

# --- Architecture (IDENTICAL for all three models) ---
EMBED_DIM   = 64              # Word embedding dimension (input representation size)
HIDDEN_SIZE = 128             # Number of hidden units in the recurrent layer
NUM_LAYERS  = 1               # Stack depth (1 = single recurrent layer)
DROPOUT     = 0.2             # Dropout probability (applied after recurrent layer)

# --- Training ---
NUM_EPOCHS     = 5          # Training iterations over the full dataset
LEARNING_RATE  = 0.001        # Adam optimizer step size
CLIP_GRAD_NORM = 5.0          # Gradient clipping threshold (critical for RNNs!)

# --- Generation ---
GEN_SEQ_LEN = 100             # Number of words to generate during evaluation

print("📐 Hyperparameters confirmed:")
print(f"   Vocabulary cap  : {VOCAB_SIZE_LIMIT:,} words")
print(f"   Sequence length : {SEQ_LEN} tokens")
print(f"   Batch size      : {BATCH_SIZE}")
print(f"   Embedding dim   : {EMBED_DIM}")
print(f"   Hidden size     : {HIDDEN_SIZE}")
print(f"   Layers          : {NUM_LAYERS}")
print(f"   Dropout         : {DROPOUT}")
print(f"   Epochs          : {NUM_EPOCHS}")
print(f"   Learning rate   : {LEARNING_RATE}")
print(f"   Gradient clip   : {CLIP_GRAD_NORM}")

📐 Hyperparameters confirmed:
   Vocabulary cap  : 5,000 words
   Sequence length : 30 tokens
   Batch size      : 64
   Embedding dim   : 64
   Hidden size     : 128
   Layers          : 1
   Dropout         : 0.2
   Epochs          : 5
   Learning rate   : 0.001
   Gradient clip   : 5.0


In [23]:
import os
import urllib.request

DATA_URL  = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = os.path.join("data", "shakespeare.txt")

# Create data directory if it doesn't exist
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)

def download_shakespeare(url: str, path: str) -> str:
    """Download Shakespeare corpus if not already cached locally."""
    if not os.path.exists(path):
        print(f"⬇️  Downloading Shakespeare corpus from GitHub...")
        urllib.request.urlretrieve(url, path)
        print(f"Saved to: {path}")
    else:
        print(f"Using cached file: {path}")

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    print(f"   Total characters : {len(text):,}")
    print(f"   Total lines      : {text.count(chr(10)):,}")
    return text

raw_text = download_shakespeare(DATA_URL, DATA_PATH)
print(f"\n📄 First 300 characters:")
print("-" * 60)
print(raw_text[:300])
print("-" * 60)

Using cached file: data\shakespeare.txt
   Total characters : 1,115,394
   Total lines      : 40,000

📄 First 300 characters:
------------------------------------------------------------
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us
------------------------------------------------------------


In [24]:
# ── Text Preprocessing ────────────────────────────────────────────────────
def preprocess_text(text: str) -> list:
    """
    Clean and tokenize raw text into a list of lowercase word tokens.
    
    Steps:
    1. Lowercase everything (reduces vocabulary size)
    2. Replace newlines with spaces (treat as continuous text)
    3. Remove non-alphabetic characters except apostrophes (contractions)
    4. Collapse multiple spaces
    5. Split into word tokens
    """
    # Step 1: Lowercase
    text = text.lower()
    
    # Step 2: Normalize newlines → spaces
    text = text.replace("\n", " ")
    
    # Step 3: Keep letters and apostrophes (for "don't", "i'll", etc.)
    text = re.sub(r"[^a-z' ]", " ", text)
    
    # Step 4: Collapse multiple consecutive spaces into one
    text = re.sub(r"\s+", " ", text).strip()
    
    # Step 5: Tokenize by splitting on whitespace
    tokens = text.split()
    
    return tokens

tokens = preprocess_text(raw_text)

print(f" Preprocessing complete:")
print(f"   Total tokens        : {len(tokens):,}")
print(f"   Unique word types   : {len(set(tokens)):,}")
print(f"\n🔍 Sample tokens (first 20):")
print(tokens[:20])

 Preprocessing complete:
   Total tokens        : 204,062
   Unique word types   : 12,631

🔍 Sample tokens (first 20):
['first', 'citizen', 'before', 'we', 'proceed', 'any', 'further', 'hear', 'me', 'speak', 'all', 'speak', 'speak', 'first', 'citizen', 'you', 'are', 'all', 'resolved', 'rather']


In [25]:
# ── Build Vocabulary from Token Frequencies ───────────────────────────────
def build_vocabulary(tokens: list, max_size: int) -> tuple:
    """
    Build word-to-index and index-to-word mappings.
    
    Special tokens:
        <UNK>  : Unknown words (those outside the vocabulary)
        <PAD>  : Padding token (reserved, index 0)
    
    Returns:
        word2idx : dict mapping word → integer index
        idx2word : dict mapping integer index → word
        vocab    : sorted list of vocabulary words
    """
    # Count all word frequencies
    freq = Counter(tokens)
    
    # Keep only the most frequent words
    most_common = freq.most_common(max_size - 2)  # -2 for special tokens
    
    # Build mappings: start at index 2 (0=PAD, 1=UNK)
    word2idx = {"<PAD>": 0, "<UNK>": 1}
    idx2word = {0: "<PAD>", 1: "<UNK>"}
    
    for idx, (word, count) in enumerate(most_common, start=2): # gives indexing starting from 2
        word2idx[word] = idx
        idx2word[idx]  = word
    
    vocab = list(word2idx.keys())
    
    print(f" Vocabulary built:")
    print(f"   Vocabulary size        : {len(word2idx):,} words")
    print(f"   Most common word       : '{most_common[0][0]}' ({most_common[0][1]:,} times)")
    print(f"   Least common in vocab  : '{most_common[-1][0]}' ({most_common[-1][1]:,} times)")
    
    # Coverage: what % of tokens are in-vocabulary?
    in_vocab = sum(1 for t in tokens if t in word2idx)
    print(f"   Vocabulary coverage    : {100*in_vocab/len(tokens):.1f}% of all tokens")
    
    return word2idx, idx2word, vocab

word2idx, idx2word, vocab = build_vocabulary(tokens, VOCAB_SIZE_LIMIT)
ACTUAL_VOCAB_SIZE = len(word2idx)

# Convert all tokens to indices (replace OOV words with <UNK>=1)
encoded = [word2idx.get(t, 1) for t in tokens]

print(f"\n Encoded sequence preview (first 15):")
print("Words  :", tokens[:15])
print("Indices:", encoded[:15])


 Vocabulary built:
   Vocabulary size        : 5,000 words
   Most common word       : 'the' (6,283 times)
   Least common in vocab  : 'steterat' (3 times)
   Vocabulary coverage    : 95.3% of all tokens

 Encoded sequence preview (first 15):
Words  : ['first', 'citizen', 'before', 'we', 'proceed', 'any', 'further', 'hear', 'me', 'speak', 'all', 'speak', 'speak', 'first', 'citizen']
Indices: [89, 270, 140, 36, 969, 144, 669, 128, 16, 103, 34, 103, 103, 89, 270]


In [26]:
# ── PyTorch Dataset: Sliding Window Sequences ─────────────────────────────
class TextDataset(Dataset):
    """
    Creates overlapping (input, target) pairs from an encoded token sequence.
    
    For next-word prediction:
        input  : tokens[i : i + SEQ_LEN]        (30 words)
        target : tokens[i + 1 : i + SEQ_LEN + 1]  (same window, shifted by 1)
    
    The model learns: given these 30 words, predict the next word at each step.
    """
    def __init__(self, encoded_tokens: list, seq_len: int):
        self.data    = torch.tensor(encoded_tokens, dtype=torch.long)
        self.seq_len = seq_len
        # Number of valid windows we can extract
        self.n_seqs  = len(encoded_tokens) - seq_len
    
    def __len__(self) -> int:
        """Total number of (input, target) sequence pairs."""
        return self.n_seqs
    
    def __getitem__(self, idx: int) -> tuple:
        """Return one (input_seq, target_seq) pair."""
        x = self.data[idx       : idx + self.seq_len    ]  # Input sequence
        y = self.data[idx + 1   : idx + self.seq_len + 1]  # Target (shifted right by 1)
        return x, y

# ── Train / Validation Split ───────────────────────────────────────────────
n_total = len(encoded)
n_val   = int(n_total * VAL_SPLIT)       # 10% for validation
n_train = n_total - n_val                # 90% for training

train_encoded = encoded[:n_train]        # First 90% = training
val_encoded   = encoded[n_train:]        # Last 10% = validation

# We DON'T shuffle the encoded sequence before splitting —
# language models need temporally ordered data!
print(f" Data splits:")
print(f"   Training tokens    : {len(train_encoded):,}")
print(f"   Validation tokens  : {len(val_encoded):,}")

# ── Create Dataset Objects ─────────────────────────────────────────────────
train_dataset = TextDataset(train_encoded, SEQ_LEN)
val_dataset   = TextDataset(val_encoded,   SEQ_LEN)

print(f"\n   Training sequences : {len(train_dataset):,}")
print(f"   Validation seqs    : {len(val_dataset):,}")

# ── Preview One Sample ─────────────────────────────────────────────────────
x_sample, y_sample = train_dataset[0]
print(f"\n Sample sequence:")
print(f"   Input  ({SEQ_LEN} tokens): {x_sample[:8].tolist()}... → words: {[idx2word[i.item()] for i in x_sample[:8]]}")
print(f"   Target ({SEQ_LEN} tokens): {y_sample[:8].tolist()}... → words: {[idx2word[i.item()] for i in y_sample[:8]]}")


 Data splits:
   Training tokens    : 183,656
   Validation tokens  : 20,406

   Training sequences : 183,626
   Validation seqs    : 20,376

 Sample sequence:
   Input  (30 tokens): [89, 270, 140, 36, 969, 144, 669, 128]... → words: ['first', 'citizen', 'before', 'we', 'proceed', 'any', 'further', 'hear']
   Target (30 tokens): [270, 140, 36, 969, 144, 669, 128, 16]... → words: ['citizen', 'before', 'we', 'proceed', 'any', 'further', 'hear', 'me']


In [27]:
# ── DataLoaders: Batching and Shuffling ───────────────────────────────────
train_loader = DataLoader(
    train_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = True,           # Shuffle SEQUENCES (not tokens within sequences)
    num_workers = 0,              # 0 = main process only (safest cross-platform)
    pin_memory  = (DEVICE.type == "cuda"),  # Faster GPU transfer when using CUDA
    drop_last   = True            # Drop incomplete final batch for stable training
)

val_loader = DataLoader(
    val_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = False,          # Never shuffle validation — deterministic evaluation
    num_workers = 0,
    pin_memory  = (DEVICE.type == "cuda"),
    drop_last   = False           # Keep all validation data for accurate metrics
)

print(f" DataLoaders created:")
print(f"   Training batches   : {len(train_loader):,}  (× {BATCH_SIZE} seq/batch)")
print(f"   Validation batches : {len(val_loader):,}")
print(f"   pin_memory         : {DEVICE.type == 'cuda'}")


 DataLoaders created:
   Training batches   : 2,869  (× 64 seq/batch)
   Validation batches : 319
   pin_memory         : True


In [28]:
# ── Base Class: Shared Architecture for RNN / LSTM / GRU ─────────────────
class RecurrentLM(nn.Module):
    """
    A language model built around a recurrent layer.
    Architecture:
        Embedding → Recurrent Layer (RNN/LSTM/GRU) → Dropout → Linear
    
    Parameters
    ----------
    vocab_size   : Size of the vocabulary (input & output dimension)
    embed_dim    : Dimension of word embeddings
    hidden_size  : Number of units in the recurrent layer
    num_layers   : Number of stacked recurrent layers
    dropout      : Dropout probability
    rnn_type     : One of 'RNN', 'LSTM', 'GRU'
    """
    def __init__(
        self,
        vocab_size  : int,
        embed_dim   : int,
        hidden_size : int,
        num_layers  : int,
        dropout     : float,
        rnn_type    : str
    ):
        super(RecurrentLM, self).__init__()  # Initialize nn.Module
        
        self.rnn_type    = rnn_type
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        
        # ── Layer 1: Embedding ─────────────────────────────────────────────
        # Maps integer word indices → dense float vectors of size embed_dim
        # padding_idx=0: the <PAD> token always maps to a zero vector
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim  = embed_dim,
            padding_idx    = 0
        )
        
        # ── Layer 2: Recurrent Layer (THE KEY DIFFERENCE) ─────────────────
        if rnn_type == "RNN":
            self.rnn = nn.RNN(
                input_size  = embed_dim,
                hidden_size = hidden_size,
                num_layers  = num_layers,
                batch_first = True,    # Input shape: (batch, seq_len, features)
                dropout     = dropout if num_layers > 1 else 0.0,
                nonlinearity= "tanh"   # Vanilla RNN uses tanh activation
            )
        elif rnn_type == "LSTM":
            self.rnn = nn.LSTM(
                input_size  = embed_dim,
                hidden_size = hidden_size,
                num_layers  = num_layers,
                batch_first = True,
                dropout     = dropout if num_layers > 1 else 0.0
                # LSTM has no 'nonlinearity' param — uses sigmoid+tanh internally
            )
        elif rnn_type == "GRU":
            self.rnn = nn.GRU(
                input_size  = embed_dim,
                hidden_size = hidden_size,
                num_layers  = num_layers,
                batch_first = True,
                dropout     = dropout if num_layers > 1 else 0.0
                # GRU has no 'nonlinearity' param — uses sigmoid+tanh internally
            )
        else:
            raise ValueError(f"Unknown rnn_type: {rnn_type}. Choose 'RNN', 'LSTM', or 'GRU'.")
        
        # ── Layer 3: Dropout (applied to recurrent output) ─────────────────
        self.dropout = nn.Dropout(p=dropout)
        
        # ── Layer 4: Output Projection ─────────────────────────────────────
        # Maps hidden_size → vocab_size (one logit per word in vocabulary)
        self.fc = nn.Linear(hidden_size, vocab_size)
        
        # ── Weight Initialization ──────────────────────────────────────────
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights for stable training."""
        # Embedding: uniform in [-0.1, 0.1]
        nn.init.uniform_(self.embedding.weight, -0.1, 0.1)
        # Output layer: uniform in [-0.1, 0.1]
        nn.init.uniform_(self.fc.weight, -0.1, 0.1)
        nn.init.zeros_(self.fc.bias)
    
    def forward(self, x: torch.Tensor, hidden=None):
        """
        Forward pass through the language model.
        
        Parameters
        ----------
        x      : (batch_size, seq_len) — batch of integer token sequences
        hidden : previous hidden state (None = zeros at start of sequence)
        
        Returns
        -------
        logits : (batch_size, seq_len, vocab_size) — unnormalized word scores
        hidden : updated hidden state (for stateful generation)
        """
        # Step 1: Embed integer tokens → dense vectors
        # Shape: (batch, seq_len) → (batch, seq_len, embed_dim)
        emb = self.embedding(x)
        emb = self.dropout(emb)   # Dropout on embeddings for regularization
        
        # Step 2: Run through recurrent layer
        # LSTM returns (output, (h_n, c_n)) — tuple of hidden + cell state
        # RNN and GRU return (output, h_n) — just hidden state
        out, hidden = self.rnn(emb, hidden)
        
        # Step 3: Apply dropout to recurrent output
        out = self.dropout(out)   # Shape: (batch, seq_len, hidden_size)
        
        # Step 4: Project to vocabulary logits
        # (batch, seq_len, hidden_size) → (batch, seq_len, vocab_size)
        logits = self.fc(out)
        
        return logits, hidden
    
    def init_hidden(self, batch_size: int):
        """Create zero initial hidden state for the start of a new sequence."""
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(DEVICE)
        if self.rnn_type == "LSTM":
            c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(DEVICE)
            return (h0, c0)   # LSTM needs BOTH h and c
        return h0              # RNN and GRU need only h

print(" RecurrentLM class defined")


 RecurrentLM class defined


In [29]:
# ── Instantiate All Three Models ──────────────────────────────────────────
def create_model(rnn_type: str) -> RecurrentLM:
    """Create and move a model to the target device."""
    model = RecurrentLM(
        vocab_size  = ACTUAL_VOCAB_SIZE,
        embed_dim   = EMBED_DIM,
        hidden_size = HIDDEN_SIZE,
        num_layers  = NUM_LAYERS,
        dropout     = DROPOUT,
        rnn_type    = rnn_type
    ).to(DEVICE)
    return model

# Reset seeds before each model creation for identical initialization baseline
torch.manual_seed(SEED)
rnn_model  = create_model("RNN")

torch.manual_seed(SEED)
lstm_model = create_model("LSTM")

torch.manual_seed(SEED)
gru_model  = create_model("GRU")

# ── Parameter Count Comparison ─────────────────────────────────────────────
def count_parameters(model: nn.Module) -> int:
    """Count total trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

rnn_params  = count_parameters(rnn_model)
lstm_params = count_parameters(lstm_model)
gru_params  = count_parameters(gru_model)

print(" Model Parameter Counts:")
print(f"   Vanilla RNN : {rnn_params:>10,} parameters")
print(f"   LSTM        : {lstm_params:>10,} parameters  ({lstm_params/rnn_params:.1f}× RNN)")
print(f"   GRU         : {gru_params:>10,} parameters  ({gru_params/rnn_params:.1f}× RNN)")

print(f"\n  RNN Architecture:")
print(rnn_model)


# ── Loss Function ─────────────────────────────────────────────────────────
# CrossEntropyLoss = log-softmax + negative log-likelihood
# It expects:
#   input  : (N, vocab_size) — logits (unnormalized scores)
#   target : (N,) — true word indices (integers)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # ignore_index=0 skips <PAD> tokens

# ── Why CrossEntropyLoss for language modeling? ────────────────────────────
# At each timestep, the model outputs vocab_size logit scores.
# CrossEntropyLoss computes: -log(softmax(logit_for_correct_word))
# This penalizes the model proportionally to how surprised it was by the true word.
# Average loss over all timesteps = negative log-likelihood per token.
# Perplexity = exp(average_loss) — the primary language model metric.

print(" Loss function: CrossEntropyLoss (ignore_index=0 for <PAD>)")
print("   Perplexity = exp(CrossEntropyLoss)")


# ── Optimizer Factory ─────────────────────────────────────────────────────
def make_optimizer(model: nn.Module) -> optim.Adam:
    """
    Create an Adam optimizer for a model.
    Adam (Adaptive Moment Estimation) is the standard choice for RNN-based LMs:
    - Maintains per-parameter learning rate adjustments
    - Handles sparse gradients well (important for embeddings)
    - Much more stable than vanilla SGD for recurrent networks
    """
    return optim.Adam(model.parameters(), lr=LEARNING_RATE)

rnn_optimizer  = make_optimizer(rnn_model)
lstm_optimizer = make_optimizer(lstm_model)
gru_optimizer  = make_optimizer(gru_model)

print(" Optimizers created: Adam (lr={})".format(LEARNING_RATE))



 Model Parameter Counts:
   Vanilla RNN :    989,832 parameters
   LSTM        :  1,064,328 parameters  (1.1× RNN)
   GRU         :  1,039,496 parameters  (1.1× RNN)

  RNN Architecture:
RecurrentLM(
  (embedding): Embedding(5000, 64, padding_idx=0)
  (rnn): RNN(64, 128, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=128, out_features=5000, bias=True)
)
 Loss function: CrossEntropyLoss (ignore_index=0 for <PAD>)
   Perplexity = exp(CrossEntropyLoss)
 Optimizers created: Adam (lr=0.001)


In [30]:
# ── Single Epoch Training Function ────────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion, device):
    """
    Train model for one full epoch over the DataLoader.
    
    Returns
    -------
    avg_loss   : float — average cross-entropy loss over all batches
    perplexity : float — exp(avg_loss)
    epoch_time : float — seconds elapsed
    """
    model.train()       # Enable dropout, batch norm updates (training mode)
    total_loss = 0.0
    n_batches  = 0
    start_time = time.time()
    
    for x_batch, y_batch in loader:
        # ── Move data to device (GPU/MPS/CPU) ─────────────────────────────
        x_batch = x_batch.to(device)  # Shape: (batch, seq_len)
        y_batch = y_batch.to(device)  # Shape: (batch, seq_len)
        
        # ── Zero gradients from previous step ─────────────────────────────
        optimizer.zero_grad()
        
        # ── Forward pass ──────────────────────────────────────────────────
        # hidden=None → model creates zero hidden state internally
        logits, _ = model(x_batch, hidden=None)
        # logits shape: (batch, seq_len, vocab_size)
        
        # ── Reshape for loss computation ───────────────────────────────────
        # CrossEntropyLoss needs: logits=(N, vocab_size), targets=(N,)
        # We flatten batch × seq_len into N:
        batch_size, seq_len, vocab_size = logits.shape
        logits_flat  = logits.view(batch_size * seq_len, vocab_size)
        targets_flat = y_batch.view(batch_size * seq_len)
        
        # ── Compute loss ───────────────────────────────────────────────────
        loss = criterion(logits_flat, targets_flat)
        
        # ── Backward pass (compute gradients) ─────────────────────────────
        loss.backward()
        
        # ── Gradient clipping (CRITICAL for vanilla RNN stability) ─────────
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD_NORM)
        
        # ── Update weights ─────────────────────────────────────────────────
        optimizer.step()
        
        total_loss += loss.item()
        n_batches  += 1
    
    avg_loss   = total_loss / n_batches
    perplexity = math.exp(avg_loss)
    epoch_time = time.time() - start_time
    
    return avg_loss, perplexity, epoch_time

print(" train_epoch() function defined")
# ── Validation Evaluation Function ────────────────────────────────────────
def evaluate(model, loader, criterion, device):
    """
    Evaluate model on a DataLoader without computing gradients.
    
    Returns
    -------
    avg_loss   : float — average cross-entropy loss
    perplexity : float — exp(avg_loss)
    """
    model.eval()        # Disable dropout for evaluation (deterministic output)
    total_loss = 0.0
    n_batches  = 0
    
    with torch.no_grad():  # Disable gradient computation → saves memory & time
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            
            logits, _ = model(x_batch, hidden=None)
            
            batch_size, seq_len, vocab_size = logits.shape
            logits_flat  = logits.view(batch_size * seq_len, vocab_size)
            targets_flat = y_batch.view(batch_size * seq_len)
            
            loss = criterion(logits_flat, targets_flat)
            total_loss += loss.item()
            n_batches  += 1
    
    avg_loss   = total_loss / n_batches
    perplexity = math.exp(min(avg_loss, 100))  # Cap to prevent overflow at inf
    
    return avg_loss, perplexity

print(" evaluate() function defined")
print("   Key difference from train: model.eval() + torch.no_grad()")


 train_epoch() function defined
 evaluate() function defined
   Key difference from train: model.eval() + torch.no_grad()


In [31]:
# ── Training History Storage ───────────────────────────────────────────────
history = {
    "RNN":  {"train_loss": [], "val_loss": [], "train_ppl": [], "val_ppl": [], "epoch_times": []},
    "LSTM": {"train_loss": [], "val_loss": [], "train_ppl": [], "val_ppl": [], "epoch_times": []},
    "GRU":  {"train_loss": [], "val_loss": [], "train_ppl": [], "val_ppl": [], "epoch_times": []},
}

# ── Master Training Function ───────────────────────────────────────────────
def train_model(model, optimizer, model_name: str, num_epochs: int):
    """
    Full training loop for one model across all epochs.
    Logs and stores training/validation metrics in the global `history` dict.
    """
    print(f"\n{'='*60}")
    print(f"  Training {model_name}")
    print(f"{'='*60}")
    print(f"  {'Epoch':>5} | {'Train Loss':>10} | {'Train PPL':>10} | {'Val Loss':>9} | {'Val PPL':>8} | {'Time':>6}")
    print(f"  {'-'*60}")
    
    for epoch in range(1, num_epochs + 1):
        # ── Train for one epoch ────────────────────────────────────────────
        tr_loss, tr_ppl, elapsed = train_epoch(
            model, train_loader, optimizer, criterion, DEVICE
        )
        # ── Evaluate on validation set ─────────────────────────────────────
        va_loss, va_ppl = evaluate(model, val_loader, criterion, DEVICE)
        
        # ── Store metrics ──────────────────────────────────────────────────
        history[model_name]["train_loss"].append(tr_loss)
        history[model_name]["val_loss"].append(va_loss)
        history[model_name]["train_ppl"].append(tr_ppl)
        history[model_name]["val_ppl"].append(va_ppl)
        history[model_name]["epoch_times"].append(elapsed)
        
        # ── Log every epoch ────────────────────────────────────────────────
        print(f"  {epoch:>5} | {tr_loss:>10.4f} | {tr_ppl:>10.2f} | {va_loss:>9.4f} | {va_ppl:>8.2f} | {elapsed:>5.1f}s")
    
    best_val_ppl = min(history[model_name]["val_ppl"])
    print(f"\n  {model_name} training complete — Best Val Perplexity: {best_val_ppl:.2f}")

print(" train_model() function ready. Starting training now...")
print(f"   Epochs       : {NUM_EPOCHS}")
print(f"   Device       : {DEVICE}")
print(f"   Batch size   : {BATCH_SIZE}")
print(f"   Train batches: {len(train_loader)}")


 train_model() function ready. Starting training now...
   Epochs       : 5
   Device       : cuda
   Batch size   : 64
   Train batches: 2869


In [32]:
# ── Train Vanilla RNN ──────────────────────────────────────────────────────
train_model(rnn_model, rnn_optimizer, "RNN", NUM_EPOCHS)


  Training RNN
  Epoch | Train Loss |  Train PPL |  Val Loss |  Val PPL |   Time
  ------------------------------------------------------------
      1 |     5.6244 |     277.09 |    6.0063 |   405.99 |  12.3s
      2 |     4.9563 |     142.07 |    6.1722 |   479.22 |  11.7s
      3 |     4.6764 |     107.39 |    6.3575 |   576.78 |  11.8s
      4 |     4.5139 |      91.28 |    6.4936 |   660.89 |  12.0s
      5 |     4.4115 |      82.39 |    6.6106 |   742.96 |  11.8s

  RNN training complete — Best Val Perplexity: 405.99


In [33]:

# ── Train LSTM ─────────────────────────────────────────────────────────────
train_model(lstm_model, lstm_optimizer, "LSTM", NUM_EPOCHS)


  Training LSTM
  Epoch | Train Loss |  Train PPL |  Val Loss |  Val PPL |   Time
  ------------------------------------------------------------
      1 |     5.7363 |     309.93 |    6.0275 |   414.68 |  12.7s
      2 |     5.1454 |     171.63 |    6.1379 |   463.09 |  12.2s
      3 |     4.8689 |     130.18 |    6.3489 |   571.87 |  11.9s
      4 |     4.6752 |     107.26 |    6.5701 |   713.41 |  12.3s
      5 |     4.5338 |      93.11 |    6.7710 |   872.21 |  11.8s

  LSTM training complete — Best Val Perplexity: 414.68


In [34]:

# ── Train GRU ──────────────────────────────────────────────────────────────
train_model(gru_model, gru_optimizer, "GRU", NUM_EPOCHS)


  Training GRU
  Epoch | Train Loss |  Train PPL |  Val Loss |  Val PPL |   Time
  ------------------------------------------------------------
      1 |     5.6541 |     285.45 |    6.0277 |   414.75 |  11.3s
      2 |     5.0580 |     157.28 |    6.1272 |   458.14 |  11.1s
      3 |     4.7880 |     120.06 |    6.2777 |   532.54 |  11.7s
      4 |     4.5979 |      99.28 |    6.3947 |   598.66 |  11.3s
      5 |     4.4451 |      85.21 |    6.5109 |   672.40 |  11.3s

  GRU training complete — Best Val Perplexity: 414.75


In [35]:
# ── Final Metric Summary Table ────────────────────────────────────────────
print("\n" + "="*70)
print("  FINAL RESULTS SUMMARY")
print("="*70)
print(f"  {'Model':<10} | {'Final Train PPL':>15} | {'Final Val PPL':>13} | {'Best Val PPL':>12} | {'Avg Epoch (s)':>13}")
print(f"  {'-'*70}")

summary = {}
for name in ["RNN", "LSTM", "GRU"]:
    h = history[name]
    final_tr   = h["train_ppl"][-1]
    final_val  = h["val_ppl"][-1]
    best_val   = min(h["val_ppl"])
    avg_time   = sum(h["epoch_times"]) / len(h["epoch_times"])
    summary[name] = {"final_tr": final_tr, "final_val": final_val,
                     "best_val": best_val, "avg_time": avg_time}
    print(f"  {name:<10} | {final_tr:>15.2f} | {final_val:>13.2f} | {best_val:>12.2f} | {avg_time:>13.2f}")

print("="*70)

# Determine winner
winner = min(summary, key=lambda k: summary[k]["best_val"])
print(f"\n  Best Model: {winner} (lowest validation perplexity: {summary[winner]['best_val']:.2f})")



  FINAL RESULTS SUMMARY
  Model      | Final Train PPL | Final Val PPL | Best Val PPL | Avg Epoch (s)
  ----------------------------------------------------------------------
  RNN        |           82.39 |        742.96 |       405.99 |         11.90
  LSTM       |           93.11 |        872.21 |       414.68 |         12.16
  GRU        |           85.21 |        672.40 |       414.75 |         11.36

  Best Model: RNN (lowest validation perplexity: 405.99)


In [36]:
# ── Token-Level Accuracy on Validation Set ────────────────────────────────
def compute_accuracy(model, loader, device):
    """
    Compute top-1 accuracy: percentage of timesteps where argmax prediction
    matches the true next word (excluding <PAD> tokens).
    """
    model.eval()
    correct = 0
    total   = 0
    
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            
            logits, _ = model(x_batch, hidden=None)
            # logits: (batch, seq_len, vocab_size)
            
            # Get predicted token at each timestep
            preds = logits.argmax(dim=-1)  # (batch, seq_len)
            
            # Mask out <PAD> positions (index 0)
            mask = (y_batch != 0)
            
            correct += (preds[mask] == y_batch[mask]).sum().item()
            total   += mask.sum().item()
    
    return 100.0 * correct / total if total > 0 else 0.0

print("Computing top-1 accuracy on validation set (this may take a moment)...")
for name, model in [("RNN", rnn_model), ("LSTM", lstm_model), ("GRU", gru_model)]:
    acc = compute_accuracy(model, val_loader, DEVICE)
    summary[name]["accuracy"] = acc
    print(f"   {name:<10}: {acc:.2f}% top-1 accuracy")


Computing top-1 accuracy on validation set (this may take a moment)...
   RNN       : 9.91% top-1 accuracy
   LSTM      : 10.59% top-1 accuracy
   GRU       : 10.39% top-1 accuracy


In [37]:
# ── Text Generation Function ───────────────────────────────────────────────
def generate_text(
    model,
    seed_text  : str,
    word2idx   : dict,
    idx2word   : dict,
    max_words  : int = 80,
    temperature: float = 0.8,
    device     = DEVICE
) -> str:
    """
    Generate text autoregressively using the trained model.
    
    Parameters
    ----------
    seed_text   : Initial prompt (space-separated words)
    temperature : Sampling temperature.
                  < 1.0 → more focused/conservative (less diverse)
                  = 1.0 → sample directly from model distribution
                  > 1.0 → more random/creative (often incoherent)
    max_words   : Number of words to generate
    
    Algorithm: Autoregressive sampling
        1. Encode seed words → integer sequence
        2. Feed to model → get logit distribution over vocabulary
        3. Apply temperature scaling
        4. Sample from the distribution (not argmax — allows diversity)
        5. Append sampled word, repeat from step 2
    """
    model.eval()
    
    # Tokenize and encode the seed text
    seed_tokens = preprocess_text(seed_text)
    seed_idx    = [word2idx.get(w, 1) for w in seed_tokens]  # OOV → <UNK>
    
    # Start with seed indices
    current_seq = seed_idx.copy()
    generated   = seed_tokens.copy()
    hidden      = None  # Will be initialized by the model
    
    with torch.no_grad():
        for _ in range(max_words):
            # Use the last SEQ_LEN tokens as context (sliding window)
            context = current_seq[-SEQ_LEN:]
            
            # Pad if context is shorter than SEQ_LEN
            if len(context) < SEQ_LEN:
                context = [0] * (SEQ_LEN - len(context)) + context
            
            # Convert to tensor: (1, SEQ_LEN)
            x = torch.tensor([context], dtype=torch.long).to(device)
            
            # Forward pass → logits: (1, SEQ_LEN, vocab_size)
            logits, hidden = model(x, hidden)
            
            # Take logits at the LAST timestep (predicting the next word)
            last_logit = logits[0, -1, :]  # (vocab_size,)
            
            # Temperature scaling: divide logits before softmax
            # Higher T → flatter distribution → more random
            # Lower T → sharper distribution → more deterministic
            scaled_logit = last_logit / temperature
            
            # Convert to probabilities
            probs = torch.softmax(scaled_logit, dim=-1)
            
            # Sample from the distribution (multinomial sampling)
            next_idx = torch.multinomial(probs, num_samples=1).item()
            
            # Decode and append
            next_word = idx2word.get(next_idx, "<UNK>")
            generated.append(next_word)
            current_seq.append(next_idx)
    
    return " ".join(generated)

print(" generate_text() function defined")
print("   Uses temperature-scaled multinomial sampling for diverse outputs")


 generate_text() function defined
   Uses temperature-scaled multinomial sampling for diverse outputs


In [38]:
# ── Generate and Compare Text from All Three Models ───────────────────────
SEED_TEXT   = "to be or not to be"
TEMPERATURE = 0.8

print("🎭 Shakespeare Text Generation Comparison")
print(f"   Seed    : '{SEED_TEXT}'")
print(f"   Temp    : {TEMPERATURE}")
print(f"   Length  : {GEN_SEQ_LEN} words")
print("=" * 70)

generated_texts = {}

for name, model in [("RNN", rnn_model), ("LSTM", lstm_model), ("GRU", gru_model)]:
    # Fix seed for reproducible sampling across all three
    torch.manual_seed(SEED)
    
    text = generate_text(
        model       = model,
        seed_text   = SEED_TEXT,
        word2idx    = word2idx,
        idx2word    = idx2word,
        max_words   = GEN_SEQ_LEN,
        temperature = TEMPERATURE,
        device      = DEVICE
    )
    generated_texts[name] = text
    
    print(f"\n {name}:")
    print("-" * 60)
    print(text)
    print("-" * 60)


🎭 Shakespeare Text Generation Comparison
   Seed    : 'to be or not to be'
   Temp    : 0.8
   Length  : 100 words

 RNN:
------------------------------------------------------------
to be or not to be so when a day <UNK> servant i can thy father did i will bear you for my troth by you the <UNK> of the duke of york no more <UNK> than thy name hath plain <UNK> a <UNK> first of a most hanging meant a afternoon and ne'er so their <UNK> god knows me he is no remedy lucio you are not a place for he is a man could desire to hate with honour sicinius find i' the <UNK> of his own <UNK> where is my lord then so cruel death is <UNK> rivers say is banished mercutio but
------------------------------------------------------------

 LSTM:
------------------------------------------------------------
to be or not to be crowned richard's life friar laurence romeo i will thy father did greet the innocent words i am going by the <UNK> scope of the duke of york no more <UNK> than thy name hath quite <UNK